In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_finalproject")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
#carga de data frames
products_df = spark.table("catalog_finalproject.bronze.products_bz")
customers_df = spark.table("catalog_finalproject.bronze.customers_bz")
orders_df = spark.table("catalog_finalproject.bronze.orders_bz")
sales_df = spark.table("catalog_finalproject.bronze.sales_bz")

In [0]:

# =========================
# PRODUCTS (clean)
# =========================
products_clean = (
    products_df
    .drop("ingestion_date")
    .filter(
        col("product_id").isNotNull() &
        col("product_name").isNotNull() &
        col("category").isNotNull() &
        col("price").isNotNull() &
        (col("price") > 0)
    )
    .select(
        col("product_id").cast("int"),
        trim(col("product_name")).alias("product_name"),
        trim(col("description")).alias("description"),
        trim(col("category")).alias("category"),
        col("price").cast(DecimalType(10,2)).alias("price"),
        trim(col("availability")).alias("availability")
    )
)

In [0]:
products_clean.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.products_sz")

In [0]:
# =========================
# CUSTOMERS (clean)
# =========================
customers_clean = (
    customers_df
    .drop("ingestion_date")
    .filter(
        col("customer_id").isNotNull() &
        col("name").isNotNull() &
        col("email").isNotNull() &
        col("country").isNotNull()
    )
    .select(
        col("customer_id").cast("int"),
        trim(col("name")).alias("name"),
        lower(col("email")).alias("email"),
        trim(col("country")).alias("country"),
        trim(col("city")).alias("city"),
        trim(col("phone")).alias("phone"),
        to_date(col("created_date")).alias("created_date")
    )
)

customers_clean.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.customers_sz")



In [0]:
# =========================
# ORDERS (clean)
# =========================
orders_clean = (
    orders_df
    .drop("ingestion_date")
    .filter(
        col("order_id").isNotNull() &
        col("customer_id").isNotNull() &
        col("order_date").isNotNull()
    )
    .select(
        col("order_id").cast("int"),
        col("customer_id").cast("int"),
        to_date(col("order_date")).alias("order_date"),
        trim(col("channel")).alias("channel"),
        trim(col("branch_name")).alias("branch_name"),
        trim(col("state")).alias("state")
    )
)

orders_clean.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.orders_sz")


In [0]:


# =========================
# SALES (clean + referential integrity)
# =========================
sales_clean = (
    sales_df
    .drop("ingestion_date")
    .filter(
        col("sale_id").isNotNull() &
        col("order_id").isNotNull() &
        col("product_id").isNotNull() &
        col("quantity").isNotNull() &
        (col("quantity") > 0) &
        col("price").isNotNull() &
        col("total_amount").isNotNull()
    )
    .select(
        col("sale_id").cast("int"),
        col("order_id").cast("int"),
        col("product_id").cast("int"),
        col("quantity").cast("int"),
        col("price").cast(DecimalType(10,2)).alias("price"),
        col("total_amount").cast(DecimalType(12,2)).alias("total_amount")
    )
)

sales_clean = (
    sales_clean
    .join(orders_clean.select("order_id"), "order_id", "inner")
    .join(products_clean.select("product_id"), "product_id", "inner")
)

sales_clean.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.sales_sz")